In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install ultralytics

In [ ]:
import os
import yaml
import shutil
from ultralytics import YOLO

# 1. Definir tus rutas personalizadas en Google Drive[cite: 1]
drive_train = '/content/drive/MyDrive/dataset/images/train'
drive_val = '/content/drive/MyDrive/dataset/images/val'
drive_dest_folder = '/content/drive/MyDrive/proyecto_yolo'

os.makedirs(drive_train, exist_ok=True)
os.makedirs(drive_val, exist_ok=True)
os.makedirs(drive_dest_folder, exist_ok=True)

# 2. Verificar/Forzar la descarga del dataset SKU-110K primero[cite: 1]
try:
    from ultralytics.data.utils import check_det_dataset as check_dataset
except ImportError:
    from ultralytics.utils.checks import check_dataset

print("Verificando dataset original SKU-110K...")
check_dataset('SKU-110K.yaml')
print("¡Dataset original listo!")

# 3. Definimos el YAML combinando SKU-110K y tu dataset[cite: 1]
custom_yaml_path = '/content/dataset_mezclado.yaml'

configuracion = {
    'path': '/content/datasets/SKU-110K',
    'train': [
        'train.txt',
        drive_train
    ],
    'val': [
        'val.txt',
        drive_val
    ],
    'nc': 1,
    'names': {0: 'object'}
}

with open(custom_yaml_path, 'w') as f:
    yaml.dump(configuracion, f, default_flow_style=False)

print(f"Archivo de configuración combinado creado en: {custom_yaml_path}")

# 4. Iniciar el entrenamiento con Parada Temprana (Early Stopping)
model = YOLO('yolov8n.pt')

print("Iniciando entrenamiento hasta que el modelo deje de mejorar...")
results = model.train(
    data=custom_yaml_path,
    epochs=30,           # Un límite muy alto para permitir que el modelo converja
    patience=3,           # Detendrá el entrenamiento si no hay mejora tras 3 épocas consecutivas
    imgsz=640,            # Resolución de las imágenes[cite: 1]
    device=0              # Forzar GPU T4[cite: 1]
)

# 5. Exportar el mejor modelo entrenado al formato NCNN[cite: 1]
export_path = model.export(format='ncnn')
print(f"\nModelo exportado localmente en Colab: {export_path}")

# 6. Mover el modelo exportado permanentemente a tu Google Drive[cite: 1]
folder_name = os.path.basename(export_path)
drive_final_path = os.path.join(drive_dest_folder, folder_name)

if os.path.exists(drive_final_path):
    shutil.rmtree(drive_final_path)

shutil.copytree(export_path, drive_final_path)

print("\n" + "="*60)
print(f"¡ÉXITO! El modelo NCNN se ha guardado en tu Google Drive:")
print(f"Ruta: {drive_final_path}")
print("="*60)